# Mission 3: Selfie Compressor & Eigenfaces 📸

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anshid/MathMeetsML/blob/main/notebooks/03_selfie_compressor.ipynb)

> **From your syllabus**
> - *Linear Algebra*: the singular value decomposition
> - *ML-II*, Unit IV: PCA (maximum variance, dimensionality reduction)

**Your mission:** shrink a photo to a fraction of its size using only the SVD, and see exactly how much you lose.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_sample_images, load_digits

np.random.seed(0)
photo = load_sample_images().images[1]                  # a bundled sample photo (flower)
img = photo.mean(axis=2) / 255.0                         # grayscale matrix with entries in [0, 1]
print("image matrix shape:", img.shape)
plt.imshow(img, cmap="gray"); plt.axis("off"); plt.show()

> **Use your own photo (optional).** Upload a picture to Colab (folder icon on the left), then run:
> ```python
> from PIL import Image
> img = np.asarray(Image.open("my_photo.jpg").convert("L").resize((400, 300))) / 255.0
> ```

## 1. A picture is a matrix; the SVD splits it into layers
$A=U\Sigma V^\top=\sum_i\sigma_i\,u_iv_i^\top$, a sum of rank-one "layers" ordered by importance $\sigma_1\ge\sigma_2\ge\dots$

In [ ]:
U, s, Vt = np.linalg.svd(img, full_matrices=False)
print("U", U.shape, " s", s.shape, " Vt", Vt.shape)
plt.semilogy(s); plt.xlabel("index i"); plt.ylabel("singular value (log scale)"); plt.title("how fast do the layers fade?"); plt.show()

## 2. Keep only the top $k$ layers (Core)
$A_k=\sum_{i\le k}\sigma_iu_iv_i^\top = U_k\,\mathrm{diag}(s_k)\,V_k^\top$. Fill in the reconstruction.

<details><summary>Hint</summary>

`U[:, :k]` is the first `k` columns, `s[:k]` the first `k` singular values, `Vt[:k, :]` the first `k` rows. Multiply columns by `s[:k]` with `U[:, :k] * s[:k]`.
</details>

In [ ]:
def compress(U, s, Vt, k):
    return ____            # TODO: (U[:, :k] * s[:k]) @ Vt[:k, :]

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for ax, k in zip(axes, [2, 10, 40, 150]):
    ax.imshow(compress(U, s, Vt, k), cmap="gray"); ax.axis("off"); ax.set_title(f"k = {k}")
plt.show()

## 3. How much did we save, and how much did we lose?
Storage: the original needs $mn$ numbers; $A_k$ needs only $k(m+n+1)$.

> **Math box (Eckart-Young).** $A_k$ is the *best* rank-$k$ approximation in Frobenius norm, and the error is exactly
> $$\|A-A_k\|_F=\sqrt{\sigma_{k+1}^2+\sigma_{k+2}^2+\cdots}.$$
> The code below checks this identity numerically.

In [ ]:
m, n = img.shape
for k in [5, 20, 50, 100]:
    err = np.linalg.norm(img - compress(U, s, Vt, k))          # Frobenius norm
    formula = np.sqrt((s[k:] ** 2).sum())
    ratio = k * (m + n + 1) / (m * n)
    print(f"k={k:<4} error={err:8.4f}  formula={formula:8.4f}  storage={ratio:6.1%}")
    assert np.isclose(err, formula)
print("Eckart-Young identity verified ✔")

**Challenge:** find the smallest $k$ with storage under 20% where the picture still looks good to you. Post it in the group chat!

In [ ]:
ks = np.arange(1, 200)
errs = [np.sqrt((s[k:] ** 2).sum()) / np.linalg.norm(img) for k in ks]
plt.plot(ks, errs); plt.xlabel("k"); plt.ylabel("relative error"); plt.grid(alpha=.3); plt.show()

## 4. Same maths, new name: PCA (and "eigen-digits")
PCA finds the directions of maximum variance, which are the eigenvectors of the covariance matrix (*ML-II Unit IV*). Faces give "eigenfaces"; here we use handwritten digits, which is exactly the same computation.
**Core:** fill in the covariance matrix of the centred data.

In [ ]:
X = load_digits().data
mean = X.mean(axis=0)
Xc = X - mean                                   # centre the data
C = ____                                        # TODO: covariance matrix, (1/(n-1)) Xc^T Xc
vals, vecs = np.linalg.eigh(C)
order = np.argsort(vals)[::-1]
vals, vecs = vals[order], vecs[:, order]
print("top 5 eigenvalues:", np.round(vals[:5], 1))

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(13, 2.6))
axes[0].imshow(mean.reshape(8, 8), cmap="gray_r"); axes[0].set_title("mean"); axes[0].axis("off")
for k in range(5):
    axes[k + 1].imshow(vecs[:, k].reshape(8, 8), cmap="coolwarm"); axes[k + 1].set_title(f"PC {k+1}"); axes[k + 1].axis("off")
plt.show()

explained = np.cumsum(vals) / vals.sum()
plt.plot(explained, "o-"); plt.axhline(0.9, color="r", ls="--"); plt.xlabel("components"); plt.ylabel("variance explained"); plt.show()
print("components for 90% of variance:", int(np.argmax(explained >= 0.9)) + 1, "out of 64")

# SVD and PCA agree: singular values of Xc relate to the eigenvalues of C
sv = np.linalg.svd(Xc, compute_uv=False)
assert np.allclose(sv ** 2 / (len(X) - 1), vals[:len(sv)].clip(min=0), atol=1e-6)
print("PCA eigenvalues = (singular values)^2 / (n-1) ✔")

## Stretch
Project every digit onto the first 2 principal components and scatter-plot, coloured by digit. Which digits separate, which overlap?

## Quest (take-home)
Run the compressor on your own photo. Report the smallest $k$ that keeps relative error under 5% and what fraction of storage that is. Bonus: compress each RGB channel separately.

## You just used...
- the **SVD** and **Eckart-Young theorem** (best low-rank approximation)
- **PCA** as an eigenproblem on the covariance matrix (ML-II Unit IV), and its link to the SVD

## Your CV line
> *Implemented image compression via truncated SVD and PCA in NumPy, verifying the Eckart-Young error formula and the SVD-PCA equivalence numerically.*